In [ ]:
import cv2, os, shutil, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from google.colab import files
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import mean_squared_error as mse
from skimage import color

# =====================================================
# IMAGE ENHANCEMENT FUNCTIONS
# =====================================================
def apply_clahe_grayscale(img):
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    return clahe.apply(img)

def apply_clahe_lab(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    lab = cv2.merge((cl, a, b))
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

def apply_gaussian_denoising_grayscale(img):
    return cv2.GaussianBlur(img, (3, 3), 1.0)

def apply_gaussian_denoising_lab(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    den_l = cv2.GaussianBlur(l, (3, 3), 1.0)
    lab = cv2.merge((den_l, a, b))
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

# =====================================================
# LIGHTNESS ORDER ERROR (LOE)
# =====================================================
def compute_loe(original, enhanced):
    y1 = color.rgb2gray(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
    y2 = color.rgb2gray(cv2.cvtColor(enhanced, cv2.COLOR_BGR2RGB))
    N = y1.shape[0] * y1.shape[1]
    order_diff = np.abs(np.sign(y1 - y1.mean()) - np.sign(y2 - y2.mean()))
    loe_val = np.sum(order_diff) / N
    return loe_val

# =====================================================
# METRICS CALCULATION FUNCTION
# =====================================================
def calculate_metrics(original, processed):
    orig_rgb = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)
    proc_rgb = cv2.cvtColor(processed, cv2.COLOR_BGR2RGB)

    psnr_val = psnr(original, processed)
    mse_val = mse(original, processed)

    # Adaptive SSIM window
    h, w = original.shape[:2]
    win = 7 if min(h, w) >= 7 else max(3, min(h, w) // 2 * 2 - 1)
    try:
        ssim_val = ssim(orig_rgb, proc_rgb, channel_axis=-1, win_size=win)
    except TypeError:
        ssim_val = ssim(orig_rgb, proc_rgb, multichannel=True, win_size=win)

    loe_val = compute_loe(original, processed)

    return psnr_val, ssim_val, mse_val, loe_val

# =====================================================
# IMAGE UPLOAD & PROCESSING
# =====================================================
uploaded = files.upload()
filename = next(iter(uploaded))
img = cv2.imread(filename)

if img is None:
    raise ValueError("Image failed to load!")

output_dir = "/content/processed_images/"

# Clear the output directory so it only contains current run outputs
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
os.makedirs(output_dir, exist_ok=True)

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# =====================================================
# SEQUENTIAL PIPELINE
# =====================================================
# Grayscale: Gaussian -> CLAHE
den_gray = apply_gaussian_denoising_grayscale(gray)
enh_gray = apply_clahe_grayscale(den_gray)

# Color: Gaussian in LAB -> CLAHE on denoised LAB result
den_color = apply_gaussian_denoising_lab(img)
enh_color = apply_clahe_lab(den_color)

# =====================================================
# METRICS EVALUATION
# =====================================================
results = {}
results["Denoised Gray"] = calculate_metrics(
    cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR),
    cv2.cvtColor(den_gray, cv2.COLOR_GRAY2BGR)
)
results["Enhanced Gray"] = calculate_metrics(
    cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR),
    cv2.cvtColor(enh_gray, cv2.COLOR_GRAY2BGR)
)
results["Denoised Color"] = calculate_metrics(img, den_color)
results["Enhanced Color"] = calculate_metrics(img, enh_color)

cols = ["PSNR", "SSIM", "MSE", "LOE"]
df = pd.DataFrame.from_dict(results, orient="index", columns=cols)
df.reset_index(inplace=True)
df.rename(columns={"index": "Process"}, inplace=True)

df.to_csv(os.path.join(output_dir, "metrics_full.csv"), index=False)

print("\n📊 --- Image Enhancement Metrics ---")
print(df.round(4))

# =====================================================
# SAVE IMAGES
# =====================================================
# Save original and grayscale images too
cv2.imwrite(os.path.join(output_dir, "original_" + filename), img)
cv2.imwrite(os.path.join(output_dir, "gray_" + filename), gray)

cv2.imwrite(os.path.join(output_dir, "den_gray_" + filename), den_gray)
cv2.imwrite(os.path.join(output_dir, "enh_gray_" + filename), enh_gray)
cv2.imwrite(os.path.join(output_dir, "den_color_" + filename), den_color)
cv2.imwrite(os.path.join(output_dir, "enh_color_" + filename), enh_color)
print("✅ Processed images saved successfully.")

# =====================================================
# VISUALIZATION SETTINGS
# =====================================================
plt.rcParams.update({
    "font.size": 20,
    "axes.titlesize": 20,
    "axes.labelsize": 20,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 20
})

def style_axes(ax, title=None, xlabel=None, ylabel=None):
    if title:
        ax.set_title(title, fontsize=20, fontweight='bold')
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=20)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=20)
    ax.tick_params(axis='both', labelsize=20)
    ax.grid(alpha=0.3)

# =====================================================
# VISUALIZATION SECTION
# =====================================================
df.columns = df.columns.str.strip()
metrics_to_plot = [c for c in df.columns if c != "Process"]

# --- BAR CHART ---
ax = df.set_index("Process")[metrics_to_plot].plot(kind="bar", figsize=(13, 7))
style_axes(
    ax,
    title="Comparison of Image Enhancement Metrics",
    xlabel="Processing Stage",
    ylabel="Metric Value"
)
plt.xticks(rotation=15)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
bar_chart_path = os.path.join(output_dir, "metrics_bar_chart.png")
plt.savefig(bar_chart_path, dpi=300, bbox_inches='tight')
plt.close()

# --- BRIGHTNESS HISTOGRAM: ORIGINAL GRAYSCALE VS ENHANCED GRAYSCALE ---
print("\nGenerating grayscale brightness distribution chart...")

fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(gray.ravel(), bins=256, color='blue', alpha=0.5, label='Original Grayscale')
ax.hist(enh_gray.ravel(), bins=256, color='orange', alpha=0.5, label='Enhanced Grayscale')
style_axes(
    ax,
    title="Brightness Distribution: Original Grayscale vs Enhanced Grayscale",
    xlabel="Pixel Intensity",
    ylabel="Frequency"
)
ax.legend(fontsize=20)
plt.tight_layout()
brightness_hist_path = os.path.join(output_dir, "brightness_histogram_gray_vs_enh_gray.png")
plt.savefig(brightness_hist_path, dpi=300, bbox_inches='tight')
plt.close()

# --- RGB CHANNEL HISTOGRAMS FOR ENHANCED COLOR IMAGE ---
print("\nGenerating RGB channel histograms...")

red_channel = enh_color[..., 2]
green_channel = enh_color[..., 1]
blue_channel = enh_color[..., 0]

# Red
fig, ax = plt.subplots(figsize=(15, 5))
ax.hist(red_channel.ravel(), bins=256, color='red', alpha=0.7, label='Red Channel')
style_axes(ax, title="Red Channel Histogram", xlabel="Pixel Intensity", ylabel="Frequency")
ax.legend(fontsize=27)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "red_histogram.png"), dpi=300, bbox_inches='tight')
plt.close()

# Green
fig, ax = plt.subplots(figsize=(15, 5))
ax.hist(green_channel.ravel(), bins=256, color='green', alpha=0.7, label='Green Channel')
style_axes(ax, title="Green Channel Histogram", xlabel="Pixel Intensity", ylabel="Frequency")
ax.legend(fontsize=27)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "green_histogram.png"), dpi=300, bbox_inches='tight')
plt.close()

# Blue
fig, ax = plt.subplots(figsize=(15, 5))
ax.hist(blue_channel.ravel(), bins=256, color='blue', alpha=0.7, label='Blue Channel')
style_axes(ax, title="Blue Channel Histogram", xlabel="Pixel Intensity", ylabel="Frequency")
ax.legend(fontsize=27)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "blue_histogram.png"), dpi=300, bbox_inches='tight')
plt.close()

print("\n✅ Visualization completed. All charts saved.")

# =====================================================
# ZIP & DOWNLOAD
# =====================================================
shutil.make_archive("/content/processed_images", "zip", output_dir)
files.download("/content/processed_images.zip")

Saving IMG 14.png to IMG 14.png

📊 --- Image Enhancement Metrics ---
          Process     PSNR    SSIM        MSE     LOE
0   Denoised Gray  28.6220  0.9217    89.3053  0.0844
1   Enhanced Gray  17.5501  0.4293  1143.0738  0.1880
2  Denoised Color  28.2620  0.8828    97.0242  0.1065
3  Enhanced Color  19.3229  0.4774   759.9547  0.1255
✅ Processed images saved successfully.

Generating grayscale brightness distribution chart...

Generating RGB channel histograms...

✅ Visualization completed. All charts saved.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>